# Solutions · 09 · The Poisson process

Worked solutions to the exercises of [notebook 09](../notebooks/09_the_poisson_process.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats, optimize
from engstoch import poisson as pp, models, datasets, special as sf
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

Using the call-centre data, test whether the Monday profile is a scaled copy of the Tuesday-Friday profile: estimate half-hourly rates for both and test the ratio for constancy (a chi-square test on the Monday counts given the scaled profile).

In [2]:
cc = pd.read_csv(datasets.path("call_centre.csv"), comment="#")
edges = np.arange(8, 20.01, 1.0)
mon = np.array([pp.counts_in_windows(g.arrival_h, edges) for d, g in cc[cc.weekday == "Mon"].groupby("day")]).sum(axis=0)
rest = cc[(cc.weekday != "Mon") & (cc.day != 8)]
base = np.array([pp.counts_in_windows(g.arrival_h, edges) for d, g in rest.groupby("day")]).sum(axis=0)
factor = mon.sum() / 4 / (base.sum() / rest.day.nunique())
expected = base / rest.day.nunique() * 4 * factor
chi = sf.chi2_sf(np.sum((mon - expected) ** 2 / expected), len(mon) - 2)
print(f"Monday factor {factor:.3f}; chi-square test of a proportional profile: p = {chi:.3f}")
print("hourly ratio Monday / other days:", (mon / 4 / (base / rest.day.nunique())).round(3).tolist())

Monday factor 1.133; chi-square test of a proportional profile: p = 0.004
hourly ratio Monday / other days: [1.158, 1.223, 1.121, 1.163, 1.167, 0.94, 1.161, 1.219, 1.09, 1.056, 1.023, 1.22]


Given the Monday totals, a proportional profile predicts the Monday hourly counts; the chi-square test rejects
it (p = 0.004), but the rejection rests almost entirely on one hour - 13:00-14:00, with a ratio of 0.94 against
about 1.15 everywhere else. Since these data are synthetic we know the true profile is proportional, so this is
a chance fluctuation of the kind a test produces when many cells are compared. In real work the next step would
be to look at that hour on each of the four Mondays before believing in a Monday lunchtime dip; for staffing,
one factor of about 1.15 on the weekday profile is the sensible model.

## Exercise 2

Simulate a Cox process whose daily rate is the call-centre profile times a Gamma(shape 20, mean 1) day factor, and show that it reproduces the overdispersion of daily totals that a Poisson process cannot.

In [3]:
lam_int = pp.cumulative_intensity(models.call_centre_rate, 24.0, 20001)
pois = [len(pp.thinning(models.call_centre_rate, 160.0, 24.0, R(k))["times"]) for k in range(400)]
cox = []
for k in range(400):
    g = R(1000 + k)
    f = g.gamma(20, 1 / 20)
    cox.append(len(pp.thinning(lambda t: f * models.call_centre_rate(t), 160.0 * f, 24.0, g)["times"]))
print(f"Poisson days: mean {np.mean(pois):.0f}, variance/mean {np.var(pois) / np.mean(pois):.2f}")
print(f"Cox days (Gamma day factor, cv {1 / np.sqrt(20):.2f}): mean {np.mean(cox):.0f}, variance/mean {np.var(cox) / np.mean(cox):.2f}; theory 1 + mean/shape = {1 + lam_int / 20:.1f}")

Poisson days: mean 1068, variance/mean 1.06
Cox days (Gamma day factor, cv 0.22): mean 1084, variance/mean 56.14; theory 1 + mean/shape = 54.4


A random day factor F with mean 1 and variance 1/20 makes the daily count mixed Poisson: Var N = E N + Var(F)(E N)²,
so the variance-to-mean ratio is 1 + Λ/20 ≈ 56 for about 1100 calls a day. Even modest day-to-day variation of
the rate (cv 22 %) swamps Poisson noise at this volume - why staffing must plan for forecast error, not just
for randomness given the forecast.

## Exercise 3

**Implement it yourself:** simulate a non-homogeneous Poisson process by inverting the cumulative intensity of lambda(t) = 2 + sin(t) on [0, 50] with a root finder, and check the counts in [0, 10] and [10, 20] against their Poisson laws.

In [4]:
lam = lambda t: 2 + np.sin(t)
Lam = lambda t: 2 * t + 1 - np.cos(t)
e = pp.homogeneous(1.0, Lam(50.0), R(7))
times_one = np.array([optimize.brentq(lambda t: Lam(t) - ei, 0, 50) for ei in e])
counts = []
for k in range(3000):
    e = pp.homogeneous(1.0, Lam(20.0), R(100 + k))
    t = np.array([optimize.brentq(lambda s: Lam(s) - ei, 0, 20) for ei in e])
    counts.append((np.sum(t < 10), np.sum((t >= 10) & (t < 20))))
counts = np.array(counts)
for j, (a, b) in enumerate(((0, 10), (10, 20))):
    m = Lam(b) - Lam(a)
    print(f"N[{a}, {b}): mean {counts[:, j].mean():.3f}, variance {counts[:, j].var():.3f}, Lambda difference {m:.3f}")
print("correlation of the two counts (independent increments):", np.corrcoef(counts.T)[0, 1].round(3))

N[0, 10): mean 21.920, variance 21.642, Lambda difference 21.839
N[10, 20): mean 18.667, variance 18.856, Lambda difference 18.753
correlation of the two counts (independent increments): -0.041


Inverting Λ(t) = 2t + 1 − cos t at each unit-rate arrival gives the non-homogeneous process; the counts in the
two windows are Poisson with means Λ(10) − Λ(0) and Λ(20) − Λ(10) and uncorrelated. A root finder per arrival is
slow but exact; tabulating Λ and interpolating, as notebook 09 did, is the fast version.